# Cartographer Benchmark Analysis

22 repos, 17 languages, 25K files, 247K nodes, 498K edges

---

In [1]:
import json, math, textwrap
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.figsize': (12, 6),
    'axes.spines.top': False,
    'axes.spines.right': False,
})

data = json.load(open('../results.json'))
repos = data['repos']
summary = data['summary']
print(f"Loaded {len(repos)} repos")
print(f"Total: {summary['total_files']:,} files, {summary['total_nodes']:,} nodes, {summary['total_edges']:,} edges")

Loaded 22 repos
Total: 25,174 files, 246,966 nodes, 498,394 edges


## 1. Indexing Performance

Files indexed per second and total time per repo.

In [2]:
repos_sorted = sorted(repos, key=lambda r: r['duration_ms'], reverse=True)
names = [r['name'] for r in repos_sorted]
times = [r['duration_ms'] / 1000 for r in repos_sorted]
files = [r['files'] for r in repos_sorted]
throughputs = [r['files'] / max(r['duration_ms'] / 1000, 0.001) for r in repos_sorted]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(names)))
bars1 = ax1.barh(names, times, color=colors[::-1])
ax1.set_xlabel('Time (seconds)')
ax1.set_title('Indexing Time by Repository')
for bar, t in zip(bars1, times):
    ax1.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
             f'{t:.1f}s', va='center', fontsize=8)

colors2 = plt.cm.Greens(np.linspace(0.3, 0.9, len(names)))
ax2.barh(names, throughputs, color=colors2[::-1])
ax2.set_xlabel('Files / second')
ax2.set_title('Indexing Throughput')
ax2.axvline(sum(throughputs)/len(throughputs), color='red', linestyle='--', alpha=0.5, label=f'Avg: {sum(throughputs)/len(throughputs):.0f}')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('benchmark_indexing.png', bbox_inches='tight', dpi=150)
plt.show()

/tmp/ipykernel_19982/438178798.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Knowledge Graph Structure

Nodes and edges per repo. The graph preserves structure while compressing source.

In [3]:
repos_sorted = sorted(repos, key=lambda r: r['nodes'], reverse=True)
names = [r['name'] for r in repos_sorted]
nodes = [r['nodes'] for r in repos_sorted]
edges = [r['edges'] for r in repos_sorted]

x = np.arange(len(names))
w = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - w/2, nodes, w, label='Nodes', color='#4A90D9')
ax.bar(x + w/2, edges, w, label='Edges', color='#7BC47F')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=45, ha='right')
ax.set_ylabel('Count')
ax.set_title('Knowledge Graph: Nodes vs Edges')
ax.legend()

for i, (n, e) in enumerate(zip(nodes, edges)):
    ax.text(i - w/2, n + max(nodes)*0.01, str(n), ha='center', va='bottom', fontsize=7, rotation=90)

plt.tight_layout()
plt.savefig('benchmark_graph_structure.png', bbox_inches='tight', dpi=150)
plt.show()

/tmp/ipykernel_19982/2341299261.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Architecture Detection

Layers and patterns detected per repository.

In [4]:
repos_sorted = sorted(repos, key=lambda r: r.get('architecture', {}).get('layers', 0), reverse=True)
names = [r['name'] for r in repos_sorted]
layers = [r.get('architecture', {}).get('layers', 0) for r in repos_sorted]
patterns = [r.get('architecture', {}).get('patterns', 0) for r in repos_sorted]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(names))
w = 0.35

ax.bar(x - w/2, layers, w, label='Layers', color='#E8A838')
ax.bar(x + w/2, patterns, w, label='Patterns', color='#8B5CF6')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=45, ha='right')
ax.set_ylabel('Count')
ax.set_title('Architecture Detection: Layers and Patterns')
ax.legend()

for i, (l, p) in enumerate(zip(layers, patterns)):
    ax.text(i, max(layers)+0.3, f'{l}/{p}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('benchmark_architecture.png', bbox_inches='tight', dpi=150)
plt.show()

/tmp/ipykernel_19982/2194997805.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Source Compression

How many source-code tokens each knowledge graph node represents — the compression ratio.

In [5]:
for r in repos:
    src_tokens = r.get('source', {}).get('tokens_est', 0)
    r['compression_ratio'] = src_tokens / max(r['nodes'], 1)

repos_sorted = sorted(repos, key=lambda r: r['compression_ratio'], reverse=True)
names = [r['name'] for r in repos_sorted]
ratios = [r['compression_ratio'] for r in repos_sorted]
src_tokens = [r.get('source', {}).get('tokens_est', 0) for r in repos_sorted]

fig, ax1 = plt.subplots(figsize=(14, 6))

colors = plt.cm.Purples(np.linspace(0.4, 0.9, len(names)))
ax1.barh(names, ratios, color=colors[::-1])
ax1.set_xlabel('Source Tokens per Graph Node')
ax1.set_title('Compression Ratio: Source Tokens per Knowledge Graph Node')

for bar, ratio, tokens in zip(ax1.patches, ratios, src_tokens):
    ax1.text(bar.get_width() + max(ratios)*0.01, bar.get_y() + bar.get_height()/2,
             f'{ratio:.0f}:1  ({tokens//1000}K tokens)', va='center', fontsize=7)

plt.tight_layout()
plt.savefig('benchmark_compression.png', bbox_inches='tight', dpi=150)
plt.show()

print(f"Average compression: {sum(ratios)/len(ratios):.0f}:1")
print(f"Total source tokens: {summary['total_tokens_est']:,}")
print(f"Total graph nodes: {summary['total_nodes']:,}")

Average compression: 1430:1
Total source tokens: 460,011,290
Total graph nodes: 246,966


/tmp/ipykernel_19982/2969219521.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Embedding Performance

Time to generate embeddings and nodes embedded per second.

In [6]:
repos_with_embed = [r for r in repos if r.get('embed_ms', 0) > 0]
repos_sorted = sorted(repos_with_embed, key=lambda r: r['embed_ms'], reverse=True)
names = [r['name'] for r in repos_sorted]
embed_ms = [r['embed_ms'] / 1000 for r in repos_sorted]
embed_nodes = [r['embed_nodes'] for r in repos_sorted]
embed_throughput = [r['embed_nodes'] / max(r['embed_ms'] / 1000, 0.001) for r in repos_sorted]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

colors = plt.cm.Oranges(np.linspace(0.3, 0.9, len(names)))
ax1.barh(names, embed_ms, color=colors[::-1])
ax1.set_xlabel('Time (seconds)')
ax1.set_title('Embedding Generation Time')
for bar, t, n in zip(ax1.patches, embed_ms, embed_nodes):
    ax1.text(bar.get_width() + max(embed_ms)*0.01, bar.get_y() + bar.get_height()/2,
             f'{t:.0f}s ({n} nodes)', va='center', fontsize=7)

colors2 = plt.cm.Reds(np.linspace(0.3, 0.9, len(names)))
ax2.barh(names, embed_throughput, color=colors2[::-1])
ax2.set_xlabel('Nodes / second')
ax2.set_title('Embedding Throughput')
ax2.axvline(sum(embed_throughput)/len(embed_throughput), color='red', linestyle='--', alpha=0.5,
            label=f'Avg: {sum(embed_throughput)/len(embed_throughput):.0f}')
ax2.legend()

plt.tight_layout()
plt.savefig('benchmark_embedding.png', bbox_inches='tight', dpi=150)
plt.show()

/tmp/ipykernel_19982/1553681438.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Query Results

Semantic search relevance: how many of 10 natural-language questions per repo found relevant results in top-5.

In [7]:
repos_sorted = sorted(repos, key=lambda r: r.get('queries', {}).get('passed', 0) / max(r.get('queries', {}).get('total', 1), 1))
names = [r['name'] for r in repos_sorted]
q_passed = [r.get('queries', {}).get('passed', 0) for r in repos_sorted]
q_total = [r.get('queries', {}).get('total', 0) for r in repos_sorted]

fig, ax = plt.subplots(figsize=(12, 5))

for i, (n, p, t) in enumerate(zip(names, q_passed, q_total)):
    color = '#2ECC71' if p == t else '#F39C12' if p > 0 else '#E74C3C'
    ax.barh(n, p, color=color, height=0.6)
    if t > 0:
        ax.text(p + 0.05, i, f'{p}/{t}', va='center', fontsize=9)

ax.set_xlabel('Passed')
ax.set_title('Semantic Search: Relevant Queries (top-5 recall)')
ax.set_xlim(0, max(q_total) + 0.5)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ECC71', label='10/10 passed'),
    Patch(facecolor='#F39C12', label='Partial'),
    Patch(facecolor='#E74C3C', label='0/10 passed'),
]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('benchmark_queries.png', bbox_inches='tight', dpi=150)
plt.show()

total_q = sum(q_total)
total_p = sum(q_passed)
print(f"Overall query pass rate: {total_p}/{total_q} = {total_p/total_q*100:.1f}%")

Overall query pass rate: 220/220 = 100.0%


/tmp/ipykernel_19982/4119926394.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Token Cost Analysis

### Honest comparison: three scenarios

| Scenario | What happens | Tokens/query | Cost (Haiku) |
|---|---|---:|---:|
| **Full source dump** | Dump entire repo into LLM context | millions | $0.07–$4.85 |
| **Manual grep + send** | Dev finds 2–5 relevant files, copies 50 lines each | ~500 | $0.00013 |
| **Cartographer** | Semantic search retrieves top-5 graph nodes | ~150 | $0.00004 |

- Full-dump is a straw man — nobody does this. But it **is** what naive "AI for code" tools pitch.
- Manual grep is the realistic baseline. Cartographer saves **~4x on tokens** vs this.
- Cartographer's real value: **you don't need to know the codebase** to find the right context.

All costs use Haiku ($0.25/M input tokens). GPT-4o would be 10x higher.

In [8]:
HAIKU_IN = 0.25e-6

MANUAL_TOKENS = 512
CART_TOKENS = 150
cost_manual = MANUAL_TOKENS * HAIKU_IN
cost_cart = CART_TOKENS * HAIKU_IN

repos_sorted = sorted(repos, key=lambda r: r.get('source', {}).get('tokens_est', 0))
names = [r['name'] for r in repos_sorted]
src_tokens = [r.get('source', {}).get('tokens_est', 0) for r in repos_sorted]
cost_dump = [t * HAIKU_IN for t in src_tokens]

# ── Log-scale dot plot ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))

y = np.arange(len(names))

ax.scatter(cost_dump, y, s=40, color='#E74C3C', alpha=0.7, label='Full source dump', zorder=3)
ax.scatter([cost_manual] * len(names), y, s=40, color='#E8A838', alpha=0.7, label='Manual grep + 2 files', zorder=4)
ax.scatter([cost_cart] * len(names), y, s=40, color='#2ECC71', alpha=0.7, label='Cartographer top-5 nodes', zorder=5)

for i in range(len(names)):
    ax.plot([cost_cart, cost_manual, cost_dump[i]], [i, i, i],
            color='#ccc', linewidth=0.5, zorder=1)

for i, (c, n) in enumerate(zip(cost_dump, names)):
    ax.text(c * 1.3, i, f'${c:.2f}', va='center', fontsize=6, color='#C0392B')

ax.set_yticks(y)
ax.set_yticklabels(names, fontsize=8)
ax.set_xscale('log')
ax.set_xlabel('Cost per query (USD, log scale)')
ax.set_title('Cost to Answer One Question: Full Dump vs Manual vs Cartographer')
ax.legend(loc='upper left', fontsize=9, markerscale=0.8)

ax.axvspan(cost_cart * 0.7, cost_cart * 1.4, alpha=0.05, color='#2ECC71')
ax.axvspan(cost_manual * 0.7, cost_manual * 1.4, alpha=0.05, color='#E8A838')

plt.tight_layout()
plt.savefig('benchmark_token_cost.png', bbox_inches='tight', dpi=150)
plt.show()

# ── Tabular comparison ────────────────────────────────────────────
print(f"{'Repo':20s} {'Tokens':>8s} {'Full dump':>10s} {'Manual':>8s} {'Cart':>8s} {'Save':>7s}")
print('-' * 65)
for n, tk, dc in zip(names, src_tokens, cost_dump):
    print(f"{n:20s} {tk//1000:>5,}K   ${dc:<7.4f}  ${cost_manual:<.6f} ${cost_cart:<.6f}  {MANUAL_TOKENS//CART_TOKENS}x")
print()
print(f"Key takeaways:")
print(f"  - Cartographer vs full dump: up to {max(cost_dump)/cost_cart:.0f}x cheaper (not a real workflow)")
print(f"  - Cartographer vs manual selection: ~{MANUAL_TOKENS//CART_TOKENS}x cheaper")
print(f"  - Cartographer's real value: semantic retrieval for any codebase, no prior knowledge needed")
print(f"  - One-time setup: ~60s indexing + ~27m embedding (CPU). Per-query: ${cost_cart:.6f}")
print(f"  - Query embedding costs are local CPU — no API calls for retrieval")

Repo                   Tokens  Full dump   Manual     Cart    Save
-----------------------------------------------------------------
luassert               100K   $0.0251   $0.000128 $0.000037  3x
chalk                  239K   $0.0599   $0.000128 $0.000037  3x
plug                   334K   $0.0837   $0.000128 $0.000037  3x
gin                    371K   $0.0929   $0.000128 $0.000037  3x
jansson                385K   $0.0964   $0.000128 $0.000037  3x
serde                  568K   $0.1422   $0.000128 $0.000037  3x
rspec-core             815K   $0.2039   $0.000128 $0.000037  3x
flask                  869K   $0.2173   $0.000128 $0.000037  3x
monolog                896K   $0.2241   $0.000128 $0.000037  3x
mdbook               1,109K   $0.2773   $0.000128 $0.000037  3x
cats                 2,085K   $0.5214   $0.000128 $0.000037  3x
tokio                2,472K   $0.6180   $0.000128 $0.000037  3x
Humanizer            3,739K   $0.9348   $0.000128 $0.000037  3x
json                 8,413K   $2.10

/tmp/ipykernel_19982/1500307078.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Correlation: Repo Size vs Performance

How indexing time scales with file count.

In [9]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

file_counts = [r['files'] for r in repos]
index_times = [r['duration_ms'] / 1000 for r in repos]
node_counts = [r['nodes'] for r in repos]
embed_times = [r.get('embed_ms', 0) / 1000 for r in repos]

ax1.scatter(file_counts, index_times, alpha=0.7, s=60, c='#4A90D9', edgecolors='white')
z = np.polyfit(file_counts, index_times, 1)
p = np.poly1d(z)
xs = np.linspace(min(file_counts), max(file_counts), 100)
ax1.plot(xs, p(xs), '--', color='#E74C3C', alpha=0.5,
         label=f'Linear fit: {z[0]:.4f}x + {z[1]:.1f} (R²={np.corrcoef(file_counts, index_times)[0,1]**2:.2f})')
ax1.set_xlabel('Files')
ax1.set_ylabel('Indexing Time (s)')
ax1.set_title('Indexing Time vs File Count')
ax1.legend(fontsize=8)

ax2.scatter(node_counts, embed_times, alpha=0.7, s=60, c='#E8A838', edgecolors='white')
z2 = np.polyfit(node_counts, embed_times, 1)
p2 = np.poly1d(z2)
xs2 = np.linspace(min(node_counts), max(node_counts), 100)
ax2.plot(xs2, p2(xs2), '--', color='#E74C3C', alpha=0.5,
         label=f'Linear fit: {z2[0]:.4f}x + {z2[1]:.1f} (R²={np.corrcoef(node_counts, embed_times)[0,1]**2:.2f})')
ax2.set_xlabel('Nodes')
ax2.set_ylabel('Embedding Time (s)')
ax2.set_title('Embedding Time vs Node Count')
ax2.legend(fontsize=8)

for ax in [ax1, ax2]:
    for r in repos:
        x = r['files'] if ax == ax1 else r['nodes']
        y = r['duration_ms'] / 1000 if ax == ax1 else r.get('embed_ms', 0) / 1000
        if y > 0:
            ax.annotate(r['name'], (x, y), fontsize=5, alpha=0.7,
                       xytext=(3, 3), textcoords='offset points')

plt.tight_layout()
plt.savefig('benchmark_correlation.png', bbox_inches='tight', dpi=150)
plt.show()

/tmp/ipykernel_19982/1360488855.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Summary Table

All key metrics in one view.

In [10]:
print(f"{'Repo':20s} {'Files':>6s} {'Nodes':>6s} {'Edges':>6s} {'Idx(s)':>7s} {'Emb(s)':>7s} {'Arch':>6s} {'Qry':>5s} {'Tok(K)':>8s} {'Ratio':>6s}")
print('-' * 85)
for r in repos:
    a = r.get('architecture', {})
    q = r.get('queries', {})
    s = r.get('source', {})
    ratio = s.get('tokens_est', 0) // max(r['nodes'], 1)
    print(f"{r['name']:20s} {r['files']:6d} {r['nodes']:6d} {r['edges']:6d} "
          f"{r['duration_ms']/1000:7.1f} {r.get('embed_ms',0)/1000:7.1f} "
          f"{a.get('layers',0)}/{a.get('patterns',0):>3d} {q.get('passed',0)}/{q.get('total',0):>2d} "
          f"{s.get('tokens_est',0)//1000:8,d} {ratio:6,d}:1")

print()
print(f"TOTAL: {summary['total_files']:,} files, {summary['total_nodes']:,} nodes, {summary['total_edges']:,} edges")
print(f"Indexing: {summary['total_time_ms']/1000:.1f}s, Embedding: {summary['total_embed_ms']/1000:.0f}s")
print(f"Queries: {summary['queries_passed']}/{summary['queries_total']} = {summary['queries_passed']/summary['queries_total']*100:.0f}%")
print(f"Source: {summary['total_tokens_est']:,} tokens → {summary['total_nodes']:,} graph nodes = {summary['total_tokens_est']//max(summary['total_nodes'],1):,}:1 compression")

Repo                  Files  Nodes  Edges  Idx(s)  Emb(s)   Arch   Qry   Tok(K)  Ratio
-------------------------------------------------------------------------------------
Humanizer               469   5045   5042     2.3    12.3 8/  3 10/10    3,739    741:1
cats                    836   9204  11923     3.7    21.8 7/  5 10/10    2,085    226:1
chalk                    13    108    107     0.0     0.0 4/  0 10/10      239  2,218:1
django                 2356  62379 116202    10.5   109.1 11/  6 10/10   19,386    310:1
fastapi                 944  10849  17018     2.2    14.2 10/  6 10/10   15,483  1,427:1
flask                    81   1485   2180     0.3     2.2 7/  1 10/10      869    585:1
gin                      99   1759   2859     0.4     2.2 8/  5 10/10      371    211:1
hugo                    929  11841  15603     3.6    20.2 10/  6 10/10   17,124  1,446:1
jansson                  51    529   1164     0.4     1.0 5/  2 10/10      385    728:1
json                    500   20

## 10. Export to CSV

For import into external tools.

In [11]:
import csv, io

buf = io.StringIO()
w = csv.writer(buf)
w.writerow(['repo', 'files', 'nodes', 'edges', 'index_ms', 'embed_ms', 'arch_layers', 'arch_patterns',
           'queries_passed', 'queries_total', 'source_chars', 'source_tokens_est', 'source_bytes',
           'compression_ratio'])
for r in repos:
    s = r.get('source', {})
    a = r.get('architecture', {})
    q = r.get('queries', {})
    w.writerow([r['name'], r['files'], r['nodes'], r['edges'], r['duration_ms'], r.get('embed_ms', 0),
                a.get('layers', 0), a.get('patterns', 0),
                q.get('passed', 0), q.get('total', 0),
                s.get('chars', 0), s.get('tokens_est', 0), s.get('bytes', 0),
                s.get('tokens_est', 0) // max(r['nodes'], 1)])

Path('benchmark_results.csv').write_text(buf.getvalue())
print("Saved notebooks/benchmark_results.csv")
print(buf.getvalue()[:500] + '...')

Saved notebooks/benchmark_results.csv
repo,files,nodes,edges,index_ms,embed_ms,arch_layers,arch_patterns,queries_passed,queries_total,source_chars,source_tokens_est,source_bytes,compression_ratio
Humanizer,469,5045,5042,2342,12295,8,3,10,10,11217785,3739261,12299555,741
cats,836,9204,11923,3659,21822,7,5,10,10,6256215,2085405,6343951,226
chalk,13,108,107,26,34,4,0,10,10,718951,239650,750779,2218
django,2356,62379,116202,10485,109065,11,6,10,10,58159992,19386664,60322604,310
fastapi,944,10849,17018,2213,14219,10,6,10,10,46451490...


---

## Key Insights

1. **Indexing scales linearly** with file count at ~317 files/s
2. **Embedding is the bottleneck** — ~10.8 min total vs ~79.5s for indexing
3. **Average compression ratio**: each graph node represents ~1,863 source-code tokens
4. **Query relevance**: 100% (220/220) of natural-language questions find relevant results in top-5
5. **Cost savings**: Cartographer costs **$0.00004/query** vs ~**$0.00013 for manual grep+send** (~4x savings). Full-dump comparison (up to $4.85) is a straw man that doesn't reflect real workflows.
6. **Cartographer's real value**: zero-knowledge semantic retrieval — find relevant code without knowing the codebase
7. **Largest repos** (spring-boot, react, django) index in 10-20s each
8. **Architecture detection**: 5-11 layers per repo, framework detection for 17 languages